# 05 — LLM tree reasoning (vectorless)

No embeddings and no keyword search. At query time the LLM is shown the project's **document tree** — each node's `qualified_name` / `title` / `kind` / `summary` — and picks the relevant symbols; those become the results.

**Requires** `OPENAI_API_KEY` (loaded from the repo-root `.env` below). One `gpt-4o-mini` call per query — a little slower and it costs a few cents for all 10 queries.

In [ ]:
import os
from nb_helpers import load_dotenv
loaded = load_dotenv('../.env')
assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY (e.g. in ../.env)'
print('OPENAI_API_KEY loaded from .env' if loaded else 'OPENAI_API_KEY already in env')

### Index (builds the document tree the LLM reads)

In [ ]:
import sys
# Index sample_repo for this method (CLI does the heavy ingestion wiring).
# --no-inspect = read source statically (don't import the package).
!{sys.executable} -m pydocs_mcp --config configs/tree.yaml index sample_repo \
    --cache-dir .pydocs-cache/tree --force --no-inspect 2>&1 \
    | grep -E "Project:|Done:|rror" || true

### Search (Python pipeline — one LLM call per query)

In [ ]:
from nb_helpers import make_searcher, load_queries, show_results

# Build the retrieval PIPELINE in Python for this method (no CLI search).
search = make_searcher("configs/tree.yaml", cache_dir='.pydocs-cache/tree')
queries = load_queries()

In [ ]:
for q in queries:
    hits = await search(q['query'], limit=5)
    show_results(q, hits)

### Takeaway
Tree reasoning leans on the LLM's understanding of names + summaries rather than vector similarity — strong when the function's purpose is clear from its tree node, and it returns *why* via the model's pick. It depends on the document tree being indexed (and on `qualified_name` being persisted — see schema v7).